# Sentiment, Themes & Plot Analysis — SF Movies


## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re
import warnings
import json
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from wordcloud import WordCloud
from scipy import stats

for resource in ['stopwords', 'punkt', 'vader_lexicon', 'wordnet', 'punkt_tab']:
    try:
        nltk.data.find(
            f'tokenizers/{resource}' if resource.startswith('punkt')
            else f'corpora/{resource}' if resource in ('stopwords', 'wordnet')
            else f'sentiment/{resource}'
        )
    except LookupError:
        nltk.download(resource, quiet=True)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

print('All imports OK')

All imports OK


## 1. Load & prepare data

In [2]:
df_full = pd.read_csv('../sf_movies_cleaned_og.csv')

# One row per unique title for NLP (sentiment/themes live at title level)
df = df_full.drop_duplicates(subset='Title').copy().reset_index(drop=True)

# Filter for only movies
df = df[df['Kind'] == 'movie'].reset_index(drop=True)

# Parse box office to numeric
df['box_office_num'] = (
    df['Box_office']
    .fillna('')
    .str.replace(r'[\$,\s]', '', regex=True)
    .replace('', np.nan)
    .astype(float)
)

# Parse RT score ("93%" → 93.0)
df['rt_score_num'] = (
    df['Rt_score']
    .fillna('')
    .str.replace('%', '', regex=False)
    .replace('', np.nan)
    .astype(float)
)

# IMDb votes, rating, Metascore as numeric
for col in ['Imdb_rating', 'Imdb_votes', 'Metascore']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Year
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

# Oscar flag
df['won_oscar'] = df['Awards'].fillna('').apply(
    lambda x: bool(re.search(r'[Ww]on.*[Oo]scar', x))
)

# Text preprocessing helper
stop_words = set(stopwords.words('english'))
extra_stops = {
    'film', 'movie', 'story', 'one', 'two', 'also', 'however', 'become',
    'soon', 'new', 'find', 'must', 'make', 'takes', 'set', 'san', 'francisco',
    'get', 'goes', 'life', 'man', 'woman', 'young', 'day', 'time', 'people',
    'help', 'meets', 'becomes', 'years', 'world', 'old'
}
ALL_STOPS = stop_words | extra_stops

def preprocess(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'[^a-zA-Z ]', ' ', str(text).lower())
    tokens = word_tokenize(text)
    return ' '.join(t for t in tokens if t not in ALL_STOPS and len(t) > 2)

df['clean_plot'] = df['Plot'].apply(preprocess)

In [3]:
print(f'Unique titles (movies only): {len(df)}')
print(f'With box office data: {df["box_office_num"].notna().sum()}')
print(f'With Metascore: {df["Metascore"].notna().sum()}')
print(f'With RT score: {df["rt_score_num"].notna().sum()}')
print(f'With IMDb rating: {df["Imdb_rating"].notna().sum()}')
print(f'With IMDb votes: {df["Imdb_votes"].notna().sum()}')
print(f'Oscar winners: {df["won_oscar"].sum()}')

# Also print the number of unique titles with non-empty plots, since that is used for NLP
print(f'With non-empty plot: {(df["clean_plot"] != "").sum()}')

Unique titles (movies only): 291
With box office data: 202
With Metascore: 199
With RT score: 248
With IMDb rating: 286
With IMDb votes: 287
Oscar winners: 24
With non-empty plot: 291


More info about how Plots info is used in IMDB: https://help.imdb.com/article/contribution/titles/plots/G56STCKTK7ESG7CP#

* One important note to make is that Plots are submited by volunteers and users. They are verified by IMDb staff but that does not makes them invulnerable from missing or biased information.

## 2. Sentiment Scoring (VADER)

**Limitation to keep in mind:** VADER is tuned on social media text.

### Analyzer

In [4]:
analyzer = SentimentIntensityAnalyzer()

def score_plot(text):
    if pd.isna(text) or str(text).strip() == '':
        return pd.Series({'compound': np.nan, 'pos': np.nan, 'neu': np.nan, 'neg': np.nan})
    return pd.Series(analyzer.polarity_scores(str(text)))

df = pd.concat([df, df['Plot'].apply(score_plot)], axis=1)

def sentiment_label(c):
    if pd.isna(c): return 'unknown'
    if c >= 0.05: return 'positive'
    if c <= -0.05: return 'negative'
    return 'neutral'

df['sentiment_label'] = df['compound'].apply(sentiment_label)

print(df['sentiment_label'].value_counts())
print(f"\nOverall mean compound: {df['compound'].mean():.3f}")
print(f"Median: {df['compound'].median():.3f}")

# Flag suspicious misfires worth knowing about
print("\nHigh-compound-score outliers to be aware of:")
outliers = df[df['compound'].abs() > 0.95][['Title', 'compound', 'Genres']].sort_values('compound')
print(outliers.head(8).to_string(index=False))
print('...')
print(outliers.tail(5).to_string(index=False))

sentiment_label
negative    142
positive    134
neutral      15
Name: count, dtype: int64

Overall mean compound: -0.003
Median: -0.000

High-compound-score outliers to be aware of:
                             Title  compound                    Genres
                The Maltese Falcon   -0.9869   Crime, Drama, Film-Noir
                           Copycat   -0.9775  Drama, Mystery, Thriller
                       Dirty Harry   -0.9774   Action, Crime, Thriller
The Assassination of Richard Nixon   -0.9738   Biography, Crime, Drama
                        Innerspace   -0.9737 Action, Adventure, Comedy
                After the Thin Man   -0.9718    Comedy, Crime, Mystery
                        The Zodiac   -0.9712      Crime, Drama, Horror
            The Lady from Shanghai   -0.9709   Crime, Drama, Film-Noir
...
                       Title  compound                  Genres
              Dr. Dolittle 2    0.9904 Comedy, Family, Fantasy
        Under the Tuscan Sun    0.9919  Comedy, D

### Genre bar chart and intro to sentiment analysis

Note: for now the html file presents the information but is not working as intended. The expected progression is the following:

**Python part:**

Extracts poster URL, compound score and genres for Zodiac, Dr. Dolittle 2 and Babies from the dataframe
Computes mean sentiment score and film count per genre (minimum 5 films) and sorts them
Injects all that data into the HTML template as JSON

**HTML file — what it shows:**

A gradient scale bar from −1 (dark) to +1 (uplifting), visible immediately on load
Zodiac poster appears with its score dot on the scale and score number below the poster
Dr. Dolittle 2 poster appears on the other end of the scale
A question box: "Where would you place this film?" with the Babies (2010) poster shown below it
The reveal: Babies score (0.0) appears, its dot lands in the center of the scale, explanation text fades in clarifying that sentiment is about tone not quality
A transition line: "Every genre carries its own emotional fingerprint"
A horizontal bar chart where each row is a genre, bars animate from 0 to their target width, colored red (negative mean) or green (positive mean)

**The intended progression:**

Each element appears as the user scrolls down
The three posters tell a concrete story before any data is shown
The genre chart is the payoff — the user already understands the scale before seeing it applied to 20+ genres

In [5]:
# ── Extract poster URLs and scores for the 3 anchor films ─────────────────────
def get_film_data(df, title):
    row = df[df['Title'].str.lower() == title.lower()].iloc[0]
    return {
        'title':   row['Title'],
        'poster':  row['Poster'] if 'Poster' in df.columns else '',
        'compound': round(float(row['compound']), 4),
        'genres':  row['Genres'] if 'Genres' in df.columns else '',
    }

zodiac     = get_film_data(df, 'Zodiac')
dolittle   = get_film_data(df, 'Dr. Dolittle 2')
babies     = get_film_data(df, 'Babies')

# Override scores with confirmed values
zodiac['compound']   = -0.9712
dolittle['compound'] =  0.9904
babies['compound']   =  0.0000

# ── Genre sentiment data ───────────────────────────────────────────────────────
df_exploded = (
    df.dropna(subset=['Genres', 'compound'])
    .assign(genre_list=lambda d: d['Genres'].str.split(', '))
    .explode('genre_list')
    .assign(genre_list=lambda d: d['genre_list'].str.strip())
)

genre_sentiment = (
    df_exploded.groupby('genre_list')['compound']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'genre_list': 'genre'})
    .query('count >= 5')
    .sort_values('mean')
)

genre_data = [
    {
        'genre': row['genre'],
        'mean':  round(float(row['mean']), 4),
        'count': int(row['count']),
    }
    for _, row in genre_sentiment.iterrows()
]

# Check prints
print('\nAnchor films:')
print(f"Zodiac:   compound={zodiac['compound']}, genres={zodiac['genres']}")
print(f"Dolittle: compound={dolittle['compound']}, genres={dolittle['genres']}")
print('\nTop 5 genres by mean sentiment:')
print(genre_sentiment[['genre', 'mean', 'count']].head(5).to_string(index=False))




Anchor films:
Zodiac:   compound=-0.9712, genres=Crime, Drama, Mystery
Dolittle: compound=0.9904, genres=Comedy, Family, Fantasy

Top 5 genres by mean sentiment:
    genre      mean  count
  Mystery -0.648938     29
   Horror -0.567800     10
    Crime -0.555220     59
Film-Noir -0.511180     10
 Thriller -0.415709     35


In [6]:
html_genre = '''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Genre Sentiment</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    background: #0a0a0a;
    font-family: -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif;
    color: #f0ece4;
    padding: 2rem 1.5rem 3rem;
    overflow-x: hidden;
    min-height: 500vh;
  }
  .container { max-width: 680px; margin: 0 auto; }

  .section-label {
    font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase;
    color: #555550; margin-bottom: 0.5rem;
  }
  .section-title {
    font-size: 20px; font-weight: 500; color: #f0ece4; margin-bottom: 0.4rem;
  }
  .section-sub {
    font-size: 13px; color: #777770; margin-bottom: 2rem; line-height: 1.6;
  }

  /* ── Scale ── */
  .scale-wrap { margin-bottom: 2.5rem; }
  .scale-track {
    position: relative; height: 10px; border-radius: 5px;
    background: linear-gradient(to right, #c0543a 0%, #4a4a47 50%, #1a8a65 100%);
    margin-bottom: 0.5rem;
  }
  .scale-labels {
    display: flex; justify-content: space-between;
    font-size: 11px; color: #555550;
  }
  .scale-zones {
    display: flex; justify-content: space-between;
    font-size: 11px; color: #444440; margin-top: 0.3rem;
  }
  .scale-dot {
    position: absolute; top: 50%; transform: translate(-50%, -50%);
    width: 18px; height: 18px; border-radius: 50%;
    border: 2px solid #0a0a0a;
    opacity: 0; transition: opacity 0.5s ease;
  }
  .scale-dot.visible { opacity: 1; }
  .scale-dot-label {
    position: absolute; top: 18px; transform: translateX(-50%);
    font-size: 10px; white-space: nowrap; font-weight: 500;
  }

  /* ── Poster cards ── */
  .posters-wrap {
    display: flex; gap: 1.5rem; justify-content: center;
    align-items: flex-start; margin-bottom: 2rem; flex-wrap: wrap;
  }
  .poster-card {
    display: flex; flex-direction: column; align-items: center;
    width: 160px;
    opacity: 0; transform: translateY(24px);
    transition: opacity 0.6s ease, transform 0.6s ease;
  }
  .poster-card.center { width: 130px; }
  .poster-card.visible { opacity: 1; transform: translateY(0); }

  .poster-img {
    width: 100%; border-radius: 8px;
    box-shadow: 0 8px 24px rgba(0,0,0,0.7);
    aspect-ratio: 2/3; object-fit: cover;
    background: #1a1a1a; display: block;
  }
  .poster-title {
    font-size: 12px; color: #d0cdc6; font-weight: 500;
    margin-top: 0.6rem; text-align: center;
  }
  .poster-genre {
    font-size: 10px; color: #555550;
    margin-top: 0.2rem; text-align: center;
  }
  .poster-score {
    font-size: 18px; font-weight: 600;
    margin-top: 0.4rem; text-align: center;
    opacity: 0; transition: opacity 0.5s ease;
  }
  .poster-score.visible { opacity: 1; }
  .poster-tone {
    font-size: 11px; font-weight: 500; letter-spacing: 0.04em;
    margin-top: 0.2rem; text-align: center;
  }

  /* ── Question block ── */
  .question-block {
    background: #111; border: 1px solid #222;
    border-radius: 12px; padding: 1.25rem 1.5rem;
    text-align: center; margin-bottom: 2rem;
    opacity: 0; transform: translateY(16px);
    transition: opacity 0.6s ease, transform 0.6s ease;
  }
  .question-block.visible { opacity: 1; transform: translateY(0); }
  .question-text {
    font-size: 15px; color: #aaa89f; font-style: italic; margin-bottom: 0.5rem;
  }
  .scroll-hint {
    font-size: 11px; color: #444440;
    animation: bob 1.6s ease-in-out infinite;
  }
  @keyframes bob {
    0%,100% { transform: translateY(0); }
    50%      { transform: translateY(4px); }
  }

  /* ── Reveal block ── */
  .reveal-block {
    opacity: 0; transform: translateY(16px);
    transition: opacity 0.6s ease, transform 0.6s ease;
    margin-bottom: 2.5rem;
  }
  .reveal-block.visible { opacity: 1; transform: translateY(0); }
  .reveal-text {
    font-size: 13px; color: #777770; line-height: 1.65;
    text-align: center; margin-bottom: 1.5rem;
  }
  .reveal-text strong { color: #aaa89f; }

  /* ── Divider ── */
  .divider {
    height: 1px; background: #1a1a1a; margin: 2.5rem 0;
    opacity: 0; transition: opacity 0.6s ease;
  }
  .divider.visible { opacity: 1; }

  /* ── Transition block ── */
  .transition-block {
    text-align: center; padding: 1rem 0 2rem;
    opacity: 0; transform: translateY(10px);
    transition: opacity 0.6s ease, transform 0.6s ease;
  }
  .transition-block.visible { opacity: 1; transform: translateY(0); }
  .transition-block p { font-size: 14px; color: #777770; line-height: 1.65; }
  .transition-block strong { color: #aaa89f; }

  /* ── Genre chart ── */
  .genre-section { margin-top: 1rem; }
  .genre-title { font-size: 18px; font-weight: 500; color: #f0ece4; margin-bottom: 0.3rem; }
  .genre-sub   { font-size: 13px; color: #777770; margin-bottom: 1.5rem; }

  .genre-axis {
    display: grid; grid-template-columns: 130px 1fr 44px;
    gap: 10px; margin-bottom: 8px;
  }
  .genre-axis-labels {
    display: flex; justify-content: space-between;
    font-size: 10px; color: #333330;
  }

  .genre-row {
    display: grid; grid-template-columns: 130px 1fr 44px;
    align-items: center; gap: 10px; margin-bottom: 6px;
    opacity: 0; transform: translateX(-10px);
    transition: opacity 0.4s ease, transform 0.4s ease;
  }
  .genre-row.visible { opacity: 1; transform: translateX(0); }

  .genre-name { font-size: 12px; color: #9a9a94; text-align: right; }
  .genre-bar-wrap {
    position: relative; height: 20px; display: flex; align-items: center;
  }
  .genre-zero {
    position: absolute; left: 50%; top: 0;
    width: 1px; height: 100%; background: #2a2a2a; z-index: 1;
  }
  .genre-bar {
    position: absolute; height: 16px; border-radius: 2px; width: 0%;
    transition: width 0.8s cubic-bezier(0.4,0,0.2,1);
  }
  .genre-bar.neg { background: #c0543a; right: 50%; }
  .genre-bar.pos { background: #1a8a65; left: 50%; }
  .genre-count { font-size: 10px; color: #444440; }
</style>
</head>
<body>
<div class="container">

  <p class="section-label">Plot Sentiment — What does it mean?</p>
  <h2 class="section-title">Not a rating. A tone.</h2>
  <p class="section-sub">
    Each film\'s plot is read by an AI that scores its emotional tone —
    from very dark (−1) to very uplifting (+1).<br>
    Here are two real examples from San Francisco films.
  </p>

  <!-- Scale -->
  <div class="scale-wrap" id="scale-wrap">
    <div class="scale-track" id="scale-track">
      <div class="scale-dot" id="dot-zodiac"
           style="background:#c0543a; left:__ZODIAC_LEFT__">
        <div class="scale-dot-label" style="color:#c0543a;">Zodiac</div>
      </div>
      <div class="scale-dot" id="dot-dolittle"
           style="background:#1a8a65; left:__DOLITTLE_LEFT__">
        <div class="scale-dot-label" style="color:#1a8a65;">Dr. Dolittle 2</div>
      </div>
      <div class="scale-dot" id="dot-babies"
           style="background:#aaa89f; left:50%;">
        <div class="scale-dot-label" style="color:#aaa89f;">Babies</div>
      </div>
    </div>
    <div class="scale-labels">
      <span>−1.0</span><span>−0.5</span><span>0</span><span>+0.5</span><span>+1.0</span>
    </div>
    <div class="scale-zones">
      <span>Dark tone</span>
      <span style="flex:1; text-align:center;">Neutral</span>
      <span>Uplifting</span>
    </div>
  </div>

  <!-- Step 1: Zodiac -->
  <div class="posters-wrap">
    <div class="poster-card" id="card-zodiac">
      <img class="poster-img" id="img-zodiac" src="" alt="Zodiac">
      <div class="poster-title">Zodiac</div>
      <div class="poster-genre" id="genre-zodiac"></div>
      <div class="poster-score" id="score-zodiac" style="color:#c0543a;"></div>
      <div class="poster-tone" style="color:#c0543a;">Dark tone</div>
    </div>
  </div>

  <!-- Step 2: Dr Dolittle -->
  <div class="posters-wrap">
    <div class="poster-card" id="card-dolittle">
      <img class="poster-img" id="img-dolittle" src="" alt="Dr. Dolittle 2">
      <div class="poster-title">Dr. Dolittle 2</div>
      <div class="poster-genre" id="genre-dolittle"></div>
      <div class="poster-score" id="score-dolittle" style="color:#1a8a65;"></div>
      <div class="poster-tone" style="color:#1a8a65;">Uplifting tone</div>
    </div>
  </div>

  <!-- Step 3: Question + Babies -->
  <div class="question-block" id="question-block">
    <div class="question-text">"Where would you place this film on the scale?"</div>
    <div class="scroll-hint">↓ scroll to reveal</div>
  </div>

  <div class="posters-wrap">
    <div class="poster-card center" id="card-babies">
      <img class="poster-img" id="img-babies" src="" alt="Babies">
      <div class="poster-title">Babies (2010)</div>
      <div class="poster-genre" id="genre-babies"></div>
    </div>
  </div>

  <!-- Step 4: Reveal -->
  <div class="reveal-block" id="reveal-block">
    <div style="font-size:22px; font-weight:600; color:#aaa89f;
                text-align:center; margin-bottom:0.5rem;">0.0000</div>
    <div style="font-size:11px; font-weight:500; color:#aaa89f;
                text-align:center; margin-bottom:1rem; letter-spacing:0.04em;">
      Neutral — purely observational
    </div>
    <div class="reveal-text">
      No heroes, no villains, no dramatic arc.<br>
      <strong>The AI reads the words in the plot and scores the emotional tone.</strong><br>
      It is not a quality judgment — Zodiac is a critically acclaimed film.
    </div>
  </div>

  <div class="divider" id="divider1"></div>

  <!-- Transition -->
  <div class="transition-block" id="transition-block">
    <p>Every genre carries its own emotional fingerprint.<br>
    <strong>Here\'s how SF films break down across genres.</strong></p>
  </div>

  <!-- Genre chart -->
  <div class="genre-section">
    <h2 class="genre-title">Plot Sentiment by Genre</h2>
    <p class="genre-sub">Mean compound sentiment score — genres with at least 5 SF films</p>
    <div class="genre-axis">
      <div></div>
      <div class="genre-axis-labels">
        <span>−0.6</span><span>−0.3</span><span>0</span><span>+0.3</span><span>+0.6</span>
      </div>
      <div></div>
    </div>
    <div id="genre-rows"></div>
  </div>

</div>
<script>
const zodiacData   = __ZODIAC_DATA__;
const dolittleData = __DOLITTLE_DATA__;
const babiesData   = __BABIES_DATA__;
const genreData    = __GENRE_DATA__;

document.getElementById('img-zodiac').src            = zodiacData.poster;
document.getElementById('img-dolittle').src          = dolittleData.poster;
document.getElementById('img-babies').src            = babiesData.poster;
document.getElementById('genre-zodiac').textContent  = zodiacData.genres;
document.getElementById('genre-dolittle').textContent= dolittleData.genres;
document.getElementById('genre-babies').textContent  = babiesData.genres;
document.getElementById('score-zodiac').textContent  = zodiacData.compound.toFixed(4);
document.getElementById('score-dolittle').textContent= dolittleData.compound.toFixed(4);

const maxAbs = Math.max(...genreData.map(g => Math.abs(g.mean)));

function buildGenreRows() {
  const wrap = document.getElementById('genre-rows');
  genreData.forEach((g, i) => {
    const isNeg = g.mean < 0;
    const pct   = (Math.abs(g.mean) / maxAbs * 50).toFixed(2);
    const row   = document.createElement('div');
    row.className   = 'genre-row';
    row.dataset.idx = i;
    row.innerHTML =
      '<div class="genre-name">' + g.genre + '</div>' +
      '<div class="genre-bar-wrap">' +
        '<div class="genre-zero"></div>' +
        '<div class="genre-bar ' + (isNeg ? 'neg' : 'pos') + '" ' +
             'data-target="' + pct + '%" style="width:0%"></div>' +
      '</div>' +
      '<div class="genre-count">n=' + g.count + '</div>';
    wrap.appendChild(row);
  });
}

function observeGenreRows() {
  const obs = new IntersectionObserver(entries => {
    entries.forEach(entry => {
      if (!entry.isIntersecting) return;
      const row = entry.target;
      const i   = parseInt(row.dataset.idx);
      setTimeout(() => {
        row.classList.add('visible');
        const bar = row.querySelector('.genre-bar');
        if (bar) {
          requestAnimationFrame(() => requestAnimationFrame(() => {
            bar.style.width = bar.dataset.target;
          }));
        }
      }, i * 55);
      obs.unobserve(row);
    });
  }, { threshold: 0.05 });
  document.querySelectorAll('.genre-row').forEach(r => obs.observe(r));
}

function reveal(el) {
  if (el) el.classList.add('visible');
}

const triggers = [
  { id: 'card-zodiac',      pct: 0.05 },
  { id: 'dot-zodiac',       pct: 0.08 },
  { id: 'score-zodiac',     pct: 0.10 },
  { id: 'card-dolittle',    pct: 0.18 },
  { id: 'dot-dolittle',     pct: 0.21 },
  { id: 'score-dolittle',   pct: 0.23 },
  { id: 'question-block',   pct: 0.35 },
  { id: 'card-babies',      pct: 0.38 },
  { id: 'reveal-block',     pct: 0.55 },
  { id: 'dot-babies',       pct: 0.57 },
  { id: 'divider1',         pct: 0.65 },
  { id: 'transition-block', pct: 0.70 },
];

const triggered = new Set();

function onScroll() {
  const scrollTop = window.pageYOffset || document.documentElement.scrollTop;
  const totalH    = document.documentElement.scrollHeight - window.innerHeight;
  const pct       = totalH > 0 ? scrollTop / totalH : 0;
  triggers.forEach(t => {
    if (!triggered.has(t.id) && pct >= t.pct) {
      triggered.add(t.id);
      reveal(document.getElementById(t.id));
    }
  });
}

window.addEventListener('scroll', onScroll, { passive: true });

window.addEventListener('load', () => {
  buildGenreRows();
  observeGenreRows();
});

window.addEventListener('message', e => {
  if (e.data === 'startGenre') {
    triggers.forEach(t => {
      triggered.add(t.id);
      reveal(document.getElementById(t.id));
    });
  }
});
</script>
</body>
</html>
'''

def score_to_left(score):
    return f"{((score + 1) / 2 * 100):.2f}%"

html_genre_out = (html_genre
  .replace('__GENRE_DATA__',    json.dumps(genre_data))
  .replace('__ZODIAC_DATA__',   json.dumps(zodiac))
  .replace('__DOLITTLE_DATA__', json.dumps(dolittle))
  .replace('__BABIES_DATA__',   json.dumps(babies))
  .replace('__ZODIAC_LEFT__',   score_to_left(zodiac['compound']))
  .replace('__DOLITTLE_LEFT__', score_to_left(dolittle['compound']))
)

with open('genre_sentiment.html', 'w', encoding='utf-8') as f:
    f.write(html_genre_out)

print('Saved -> genre_sentiment.html')

Saved -> genre_sentiment.html


### Processing of the information for the sentiment analysis plots

It is divided in 3 splits. One for each of the plots (Oscar winners, Box office and IMDB rating)

In [9]:
bins    = 21
range_  = (-1.05, 1.05)

# Overall bars (already computed, keeping for reference)
counts_all, edges = np.histogram(df['compound'].dropna(), bins=bins, range=range_)
bar_data = [
    {'x': round((edges[i]+edges[i+1])/2, 3), 'y': int(counts_all[i]),
     'zone': 'neg' if (edges[i]+edges[i+1])/2 < -0.05 else ('neu' if (edges[i]+edges[i+1])/2 <= 0.05 else 'pos')}
    for i in range(bins)
]
label_counts = df['sentiment_label'].value_counts()
pct_all      = (label_counts / label_counts.sum() * 100).round(0).astype(int)
neg_pct      = int(pct_all.get('negative', 0))
neu_pct      = int(pct_all.get('neutral',  0))
pos_pct      = int(pct_all.get('positive', 0))

# Oscar split
oscar_yes = df[df['won_oscar']]['compound'].dropna()
oscar_no  = df[~df['won_oscar']]['compound'].dropna()
t_stat, p_value = stats.ttest_ind(oscar_yes, oscar_no)

counts_no,  edges_o = np.histogram(oscar_no,  bins=bins, range=range_)
counts_yes, _       = np.histogram(oscar_yes, bins=bins, range=range_)

oscar_no_bars = [
    {'x': round((edges_o[i]+edges_o[i+1])/2, 3), 'y': int(counts_no[i])}
    for i in range(bins)
]
oscar_yes_bars = [
    {'x': round((edges_o[i]+edges_o[i+1])/2, 3), 'y': int(counts_yes[i])}
    for i in range(bins)
]

oscar_mean     = round(float(oscar_yes.mean()), 3)
non_oscar_mean = round(float(oscar_no.mean()),  3)
p_value        = round(float(p_value), 4)
n_oscar        = int(oscar_yes.count())
n_non_oscar    = int(oscar_no.count())

# Box office split
bo_threshold = df['box_office_num'].quantile(0.75)
bo_top = df[df['box_office_num'] >= bo_threshold]['compound'].dropna()
bo_bot = df[df['box_office_num'] <  bo_threshold]['compound'].dropna()
t_bo, p_bo = stats.ttest_ind(bo_top, bo_bot)

counts_bot, edges_bo = np.histogram(bo_bot, bins=bins, range=range_)
counts_top, _        = np.histogram(bo_top, bins=bins, range=range_)

bo_bot_bars = [
    {'x': round((edges_bo[i]+edges_bo[i+1])/2, 3), 'y': int(counts_bot[i])}
    for i in range(bins)
]
bo_top_bars = [
    {'x': round((edges_bo[i]+edges_bo[i+1])/2, 3), 'y': int(counts_top[i])}
    for i in range(bins)
]

bo_top_mean = round(float(bo_top.mean()), 3)
bo_bot_mean = round(float(bo_bot.mean()), 3)
p_bo_val    = round(float(p_bo), 4)
n_bo_top    = int(bo_top.count())
n_bo_bot    = int(bo_bot.count())
bo_thresh_m = round(float(bo_threshold / 1e6), 1)

# IMDb rating split
imdb_threshold = 7.5
imdb_high = df[df['Imdb_rating'] >= imdb_threshold]['compound'].dropna()
imdb_low  = df[df['Imdb_rating'] <  imdb_threshold]['compound'].dropna()
t_imdb, p_imdb = stats.ttest_ind(imdb_high, imdb_low)

counts_imdb_high, edges_imdb = np.histogram(imdb_high, bins=bins, range=range_)
counts_imdb_low,  _          = np.histogram(imdb_low,  bins=bins, range=range_)

imdb_high_bars = [
    {'x': round((edges_imdb[i]+edges_imdb[i+1])/2, 3), 'y': int(counts_imdb_high[i])}
    for i in range(bins)
]
imdb_low_bars = [
    {'x': round((edges_imdb[i]+edges_imdb[i+1])/2, 3), 'y': int(counts_imdb_low[i])}
    for i in range(bins)
]

imdb_high_mean = round(float(imdb_high.mean()), 3)
imdb_low_mean  = round(float(imdb_low.mean()),  3)
p_imdb_val     = round(float(p_imdb), 4)
n_imdb_high    = int(imdb_high.count())
n_imdb_low     = int(imdb_low.count())

Then this is the complete html file for the sentiment analysis plots, the idea goes like:
- Floating squares above percentages of the sentiment analysis.
- Change to a histogram to show the distribution
- Change to the histogram of Oscar winners vs Not winners
- Change to the histogram of great Box Office vs Not
- Change to the histogram of high IMDb rating vs Not

In [8]:
# ── HTML Template ─────────────────────────────────────────────────────────────
html_template = '''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>SF Movie Sentiment</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    background: #0a0a0a;
    font-family: -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif;
    color: #f0ece4;
    padding: 1rem 1.5rem 1rem;
    overflow-x: hidden;
  }
  .container { max-width: 680px; margin: 0 auto; }
  h2 { font-size: 18px; font-weight: 500; color: #f0ece4; margin-bottom: 4px; }
  .subtitle { font-size: 14px; color: #aaa89f; margin-bottom: 1.25rem; }
  .legend { display: flex; gap: 20px; margin-bottom: 1.25rem; flex-wrap: wrap; }
  .legend-item {
    display: flex; align-items: center; gap: 6px;
    font-size: 14px; color: #9a9a94; transition: opacity 0.6s ease;
  }
  .legend-dot { width: 12px; height: 12px; border-radius: 50%; flex-shrink: 0; }

  #squares-stage { position: relative; width: 100%; height: 300px; }
  canvas#squaresCanvas { position: absolute; top: 0; left: 0; width: 100%; height: 100%; }

  .chart-wrap {
    position: relative; width: 100%; height: 300px;
    opacity: 0; transition: opacity 0.4s ease;
  }
  .chart-wrap.visible { opacity: 1; }

  #annotationCanvas {
    position: absolute; top: 0; left: 0; width: 100%; height: 100%;
    pointer-events: none; opacity: 0;
  }

  /* 3-card layout (first chart) */
  .cards {
    display: grid; grid-template-columns: repeat(3, 1fr);
    gap: 12px; margin-top: 1.5rem;
  }
  /* 6-card layout (comparison charts) */
  .cards-6 {
    display: grid; grid-template-columns: repeat(3, 1fr);
    gap: 12px; margin-top: 1.5rem;
  }
  .card-group-label {
    grid-column: 1 / -1;
    font-size: 11px; text-transform: uppercase;
    letter-spacing: 0.08em;
    padding: 0.35rem 0 0.15rem;
    border-top: 1px solid #1f1f1f;
    margin-top: 0.1rem;
  }
  .card-group-label.first { border-top: none; margin-top: 0; }

  .card { border-radius: 10px; padding: 0.75rem 1rem; text-align: center; }
  .card-neg { background: #1a1008; border: 1px solid #D85A30; }
  .card-neu { background: #141414; border: 1px solid #5F5E5A; }
  .card-pos { background: #0a1a12; border: 1px solid #1D9E75; }
  .card-num { font-size: 24px; font-weight: 500; margin-bottom: 4px; }
  .card-neg .card-num { color: #D85A30; }
  .card-neu .card-num { color: #9a9a94; }
  .card-pos .card-num { color: #1D9E75; }
  .card-lbl { font-size: 12px; color: #9a9a94; }

  .footer { font-size: 11px; color: #777770; font-style: italic; margin-top: 1.25rem; }
</style>
</head>
<body>
<div class="container">

  <h2 id="main-title">How do SF movie plots feel emotionally?</h2>
  <p class="subtitle" id="main-subtitle">Loading...</p>

  <div class="legend">
    <span class="legend-item" id="leg-neg"><span class="legend-dot" style="background:#F0997B;"></span>Dark / negative tone</span>
    <span class="legend-item" id="leg-neu"><span class="legend-dot" style="background:#B4B2A9;"></span>Neutral / balanced</span>
    <span class="legend-item" id="leg-pos"><span class="legend-dot" style="background:#5DCAA5;"></span>Uplifting / positive tone</span>
    <span class="legend-item" id="leg-oscar"   style="opacity:0;"><span class="legend-dot" style="background:#FFD700;"></span>Oscar winners</span>
    <span class="legend-item" id="leg-nooscar" style="opacity:0;"><span class="legend-dot" style="background:#646464;"></span>Non-winners</span>
    <span class="legend-item" id="leg-botop"   style="opacity:0;"><span class="legend-dot" style="background:#43A047;"></span>Top 25% box office</span>
    <span class="legend-item" id="leg-bobot"   style="opacity:0;"><span class="legend-dot" style="background:#646464;"></span>Bottom 75% box office</span>
    <span class="legend-item" id="leg-imdbhigh" style="opacity:0;"><span class="legend-dot" style="background:#E91E63;"></span>IMDb 7.5+</span>
    <span class="legend-item" id="leg-imdblow"  style="opacity:0;"><span class="legend-dot" style="background:#646464;"></span>IMDb below 7.5</span>
  </div>

  <div id="squares-stage">
    <canvas id="squaresCanvas"></canvas>
  </div>

  <div class="chart-wrap" id="chartWrap" style="display:none; position:relative;">
    <canvas id="sentChart"></canvas>
    <canvas id="annotationCanvas"></canvas>
  </div>

  <div class="cards" id="cards-wrap">
    <div class="card card-neg">
      <div class="card-num" id="neg-pct">-</div>
      <div class="card-lbl">Dark / negative plots</div>
    </div>
    <div class="card card-neu">
      <div class="card-num" id="neu-pct">-</div>
      <div class="card-lbl">Balanced / neutral plots</div>
    </div>
    <div class="card card-pos">
      <div class="card-num" id="pos-pct">-</div>
      <div class="card-lbl">Uplifting / positive plots</div>
    </div>
  </div>

  <p class="footer">Emotional tone rated from -1 (very dark) to +1 (very uplifting).</p>

</div>
<script>
const bars         = __BARS_JSON__;
const oscarNoBars  = __OSCAR_NO_BARS__;
const oscarYesBars = __OSCAR_YES_BARS__;
const boBotBars    = __BO_BOT_BARS__;
const boTopBars    = __BO_TOP_BARS__;
const negPct       = __NEG_PCT__;
const neuPct       = __NEU_PCT__;
const posPct       = __POS_PCT__;
const totalFilms   = __TOTAL__;
const nOscar       = __N_OSCAR__;
const nNonOscar    = __N_NON_OSCAR__;
const nBoTop       = __N_BO_TOP__;
const nBoBot       = __N_BO_BOT__;
const boThreshM    = __BO_THRESH_M__;
const imdbHighBars = __IMDB_HIGH_BARS__;
const imdbLowBars  = __IMDB_LOW_BARS__;
const nImdbHigh    = __N_IMDB_HIGH__;
const nImdbLow     = __N_IMDB_LOW__;

document.getElementById('neg-pct').textContent = negPct + '%';
document.getElementById('neu-pct').textContent = neuPct + '%';
document.getElementById('pos-pct').textContent = posPct + '%';
document.getElementById('main-subtitle').textContent =
  totalFilms + ' films scored by AI reading their plot descriptions';

const COLORS  = { neg: '#F0997B', neu: '#B4B2A9', pos: '#5DCAA5' };
const BORDERS = { neg: '#D85A30', neu: '#5F5E5A', pos: '#1D9E75' };

// ── Square cluster animation ──────────────────────────────────────────────────
const canvas = document.getElementById('squaresCanvas');
const ctx    = canvas.getContext('2d');
const SQ = 10, GAP = 3;
const total    = 60;
const negCount = Math.round(total * negPct / 100);
const neuCount = Math.round(total * neuPct / 100);
const posCount = total - negCount - neuCount;

function buildCluster(count, zone) {
  const cols = Math.ceil(Math.sqrt(count * 1.8));
  return Array.from({ length: count }, (_, i) => ({
    zone, col: i % cols, row: Math.floor(i / cols),
  }));
}
const negSquares = buildCluster(negCount, 'neg');
const neuSquares = buildCluster(neuCount, 'neu');
const posSquares = buildCluster(posCount, 'pos');
const allSquares = [...negSquares, ...neuSquares, ...posSquares];

function getClusterPositions(squares, centerX, baseY) {
  const cols   = Math.ceil(Math.sqrt(squares.length * 1.8));
  const blockW = cols * (SQ + GAP);
  const startX = centerX - blockW / 2;
  return squares.map(s => ({
    x: startX + s.col * (SQ + GAP),
    y: baseY  - s.row * (SQ + GAP),
  }));
}

function getBarPositions(squares, canvasW, canvasH) {
  const maxVal  = Math.max(...bars.map(b => b.y));
  const barW    = (canvasW - 60) / bars.length;
  const chartH  = canvasH - 40;
  const originX = 30, originY = canvasH - 40;
  const assignments = [];
  let si = 0;
  for (let bi = 0; bi < bars.length; bi++) {
    const bar   = bars[bi];
    const count = Math.round((bar.y / bars.reduce((a,b) => a+b.y, 0)) * squares.length);
    for (let k = 0; k < count && si < squares.length; k++, si++) {
      assignments.push({
        x: originX + bi * barW + (barW - SQ) / 2,
        y: originY - (k+1) * (chartH/maxVal * bar.y / Math.max(count,1)) * (1/squares.length*count),
        zone: bar.x < -0.05 ? 'neg' : bar.x <= 0.05 ? 'neu' : 'pos',
      });
    }
  }
  while (si < squares.length) {
    assignments.push({ x: canvasW/2, y: canvasH/2, zone: 'neu' }); si++;
  }
  return assignments;
}

let startPos = [], endPos = [], rafId = null, phase = 'clusters';

function resize() {
  canvas.width  = canvas.offsetWidth;
  canvas.height = canvas.offsetHeight;
  computePositions();
  if (phase === 'clusters') drawClusters();
}
function computePositions() {
  const W = canvas.width, H = canvas.height, baseY = H - 20;
  startPos = [
    ...getClusterPositions(negSquares, W * 0.20, baseY),
    ...getClusterPositions(neuSquares, W * 0.50, baseY),
    ...getClusterPositions(posSquares, W * 0.80, baseY),
  ];
  endPos = getBarPositions(allSquares, W, H);
}
function easeInOut(t) { return t < 0.5 ? 2*t*t : -1+(4-2*t)*t; }
function drawFrame(t) {
  const e = easeInOut(Math.min(t, 1));
  ctx.clearRect(0, 0, canvas.width, canvas.height);
  allSquares.forEach((sq, i) => {
    const s = startPos[i]||{x:0,y:0}, en = endPos[i]||{x:0,y:0};
    ctx.fillStyle = COLORS[sq.zone];
    ctx.beginPath();
    ctx.roundRect(s.x+(en.x-s.x)*e, s.y+(en.y-s.y)*e, SQ, SQ, 2);
    ctx.fill();
  });
}
function drawClusters() {
  ctx.clearRect(0, 0, canvas.width, canvas.height);
  [{x: canvas.width*0.20, text:'Dark',      color: COLORS.neg},
   {x: canvas.width*0.50, text:'Neutral',   color: COLORS.neu},
   {x: canvas.width*0.80, text:'Uplifting', color: COLORS.pos}
  ].forEach(l => {
    ctx.fillStyle = l.color;
    ctx.font = '12px -apple-system, sans-serif';
    ctx.textAlign = 'center';
    ctx.fillText(l.text, l.x, 16);
  });
  allSquares.forEach((sq, i) => {
    const s = startPos[i]||{x:0,y:0};
    const floatY = s.y + Math.sin(Date.now()/900 + i*0.3)*2;
    ctx.fillStyle = COLORS[sq.zone];
    ctx.beginPath();
    ctx.roundRect(s.x, floatY, SQ, SQ, 2);
    ctx.fill();
  });
}
function idleLoop() {
  if (phase !== 'clusters') return;
  drawClusters();
  rafId = requestAnimationFrame(idleLoop);
}

// ── Chart.js ──────────────────────────────────────────────────────────────────
let chartInstance = null;

function buildChart() {
  chartInstance = new Chart(document.getElementById('sentChart'), {
    type: 'bar',
    data: {
      labels: bars.map(b => b.x.toFixed(2)),
      datasets: [{
        data: bars.map(b => b.y),
        backgroundColor: bars.map(b => COLORS[b.zone]),
        borderColor:     bars.map(b => BORDERS[b.zone]),
        borderWidth: 0, borderRadius: 0, borderSkipped: false,
        categoryPercentage: 1.0, barPercentage: 1.0,
      }]
    },
    options: {
      responsive: true, maintainAspectRatio: false,
      animation: { duration: 600 },
      plugins: {
        legend: { display: false },
        tooltip: {
          callbacks: {
            title: items => {
              const s = parseFloat(items[0].label);
              const z = s < -0.05 ? 'Dark tone' : s <= 0.05 ? 'Neutral tone' : 'Uplifting tone';
              return 'Score ' + items[0].label + '  .  ' + z;
            },
            label: item => ' ' + item.raw + ' films'
          },
          backgroundColor: '#1a1a1a', titleColor: '#cfcfc4',
          bodyColor: '#cfcfc4', padding: 10, cornerRadius: 8
        }
      },
      scales: {
        x: {
          ticks: {
            autoSkip: false, maxRotation: 0,
            callback: (val, i) => {
              const label = parseFloat(bars[i].x.toFixed(2));
              return [-1,-0.5,0,0.5,1].includes(label) ? label.toFixed(1) : '';
            },
            color: '#cfcfc4', font: { size: 13 }
          },
          grid: { display: false }, border: { display: false },
          categoryPercentage: 1.0, barPercentage: 1.0
        },
        y: {
          title: { display: true, text: 'Number of films', color: '#cfcfc4', font: {size:13} },
          ticks: { color: '#cfcfc4', font: {size:13} },
          grid: { color: 'rgba(255,255,255,0.10)' },
          border: { display: false }
        }
      }
    }
  });
}

// ── Card helpers ──────────────────────────────────────────────────────────────
function calcPcts(barsArr) {
  const total = barsArr.reduce((a, b) => a + b.y, 0);
  if (total === 0) return { neg: 0, neu: 0, pos: 0 };
  let neg = 0, neu = 0, pos = 0;
  barsArr.forEach(b => {
    if      (b.x < -0.05) neg += b.y;
    else if (b.x <= 0.05) neu += b.y;
    else                  pos += b.y;
  });
  return {
    neg: Math.round(neg / total * 100),
    neu: Math.round(neu / total * 100),
    pos: Math.round(pos / total * 100),
  };
}

function updateCardsComparison(labelA, colorA, pA, labelB, colorB, pB) {
  const wrap = document.getElementById('cards-wrap');
  wrap.className = 'cards-6';
  wrap.innerHTML =
    '<div class="card-group-label first" style="color:' + colorA + '">' + labelA + '</div>' +
    '<div class="card card-neg"><div class="card-num" style="color:#D85A30">' + pA.neg + '%</div><div class="card-lbl">Negative</div></div>' +
    '<div class="card card-neu"><div class="card-num" style="color:#9a9a94">' + pA.neu + '%</div><div class="card-lbl">Neutral</div></div>' +
    '<div class="card card-pos"><div class="card-num" style="color:#1D9E75">' + pA.pos + '%</div><div class="card-lbl">Positive</div></div>' +
    '<div class="card-group-label" style="color:' + colorB + '">' + labelB + '</div>' +
    '<div class="card card-neg"><div class="card-num" style="color:#D85A30">' + pB.neg + '%</div><div class="card-lbl">Negative</div></div>' +
    '<div class="card card-neu"><div class="card-num" style="color:#9a9a94">' + pB.neu + '%</div><div class="card-lbl">Neutral</div></div>' +
    '<div class="card card-pos"><div class="card-num" style="color:#1D9E75">' + pB.pos + '%</div><div class="card-lbl">Positive</div></div>';
}

// ── Shared overlay chart builder ──────────────────────────────────────────────
function buildOverlayChart(baseBars, overlayBars, overlayColor,
                           titleText, subtitleText,
                           legBaseId, legOverlayId) {
  if (chartInstance) chartInstance.destroy();
  document.getElementById('annotationCanvas').style.opacity = '0';

  const baseTotal    = baseBars.reduce((a,b) => a+b.y, 0);
  const overlayTotal = overlayBars.reduce((a,b) => a+b.y, 0);
  const baseNorm     = baseBars.map(b    => baseTotal    > 0 ? b.y/baseTotal    : 0);
  const overlayNorm  = overlayBars.map(b => overlayTotal > 0 ? b.y/overlayTotal : 0);

  chartInstance = new Chart(document.getElementById('sentChart'), {
    type: 'bar',
    data: {
      labels: baseBars.map(b => b.x.toFixed(2)),
      datasets: [
        {
          label: 'base',
          data: baseNorm,
          backgroundColor: 'rgba(100,100,100,0.9)',
          borderColor: 'transparent',
          borderWidth: 0, borderRadius: 0, borderSkipped: false,
          categoryPercentage: 1.0, barPercentage: 1.0,
          order: 2,
        },
        {
          label: 'overlay',
          data: overlayBars.map(() => 0),
          backgroundColor: overlayColor,
          borderColor: 'transparent',
          borderWidth: 0, borderRadius: 0, borderSkipped: false,
          categoryPercentage: 1.0, barPercentage: 1.0,
          order: 1,
        }
      ]
    },
    options: {
      responsive: true, maintainAspectRatio: false,
      animation: { duration: 800 },
      plugins: {
        legend: { display: false },
        tooltip: {
          callbacks: {
            title: items => 'Score ' + items[0].label,
            label: item => ' ' + (item.raw * 100).toFixed(1) + '% of group'
          },
          backgroundColor: '#1a1a1a', titleColor: '#cfcfc4',
          bodyColor: '#cfcfc4', padding: 10, cornerRadius: 8
        }
      },
      scales: {
        x: {
          stacked: false,
          ticks: {
            autoSkip: false, maxRotation: 0,
            callback: (val, i) => {
              const label = parseFloat(baseBars[i].x.toFixed(2));
              return [-1,-0.5,0,0.5,1].includes(label) ? label.toFixed(1) : '';
            },
            color: '#cfcfc4', font: { size: 13 }
          },
          grid: { display: false }, border: { display: false },
          categoryPercentage: 1.0, barPercentage: 1.0, offset: false
        },
        y: {
          stacked: false,
          title: { display: true, text: 'Share of films in group (%)', color: '#cfcfc4', font: {size:13} },
          ticks: {
            color: '#cfcfc4', font: { size: 13 },
            callback: val => (val * 100).toFixed(0) + '%'
          },
          grid: { color: 'rgba(255,255,255,0.10)' },
          border: { display: false }
        }
      }
    },
    plugins: [{
      id: 'overlayBars',
      beforeDatasetsDraw(chart) {
        const meta0 = chart.getDatasetMeta(0);
        const meta1 = chart.getDatasetMeta(1);
        if (!meta0.data.length || !meta1.data.length) return;
        const chartWidth = chart.chartArea.right - chart.chartArea.left;
        const binWidth   = Math.ceil(chartWidth / baseBars.length) + 1;
        meta0.data.forEach((bar, i) => {
          bar.width = binWidth;
          if (meta1.data[i]) {
            meta1.data[i].x     = bar.x;
            meta1.data[i].width = binWidth;
          }
        });
      }
    }]
  });

  // Update legend
  ['leg-neg','leg-neu','leg-pos','leg-oscar','leg-nooscar','leg-botop','leg-bobot','leg-imdbhigh','leg-imdblow']
    .forEach(id => document.getElementById(id).style.opacity = '0');
  document.getElementById(legBaseId).style.opacity    = '1';
  document.getElementById(legOverlayId).style.opacity = '1';

  document.getElementById('main-title').textContent    = titleText;
  document.getElementById('main-subtitle').textContent = subtitleText;

  // Animate overlay bars up then update cards
  setTimeout(() => {
    chartInstance.data.datasets[1].data = overlayNorm;
    chartInstance.options.animation = { duration: 1200 };
    chartInstance.update('active');
  }, 900);
}

// ── Phase transitions ─────────────────────────────────────────────────────────
function morphToOscar() {
  if (!chartInstance || phase !== 'chart') return;
  phase = 'oscar';
  buildOverlayChart(
    oscarNoBars, oscarYesBars,
    'rgba(255,215,0,0.5)',
    'Plot Sentiment: Oscar Winners vs. Non-Winners',
    'Oscar winners (n=' + nOscar + ') vs. all other films (n=' + nNonOscar + ')',
    'leg-nooscar', 'leg-oscar'
  );
  setTimeout(() => {
    updateCardsComparison(
      'Non-winners',   '#9a9a94', calcPcts(oscarNoBars),
      'Oscar winners', '#FFD700', calcPcts(oscarYesBars)
    );
  }, 900);
}

function morphToBoxOffice() {
  if (phase !== 'oscar' && phase !== 'chart') return;
  phase = 'boxoffice';
  buildOverlayChart(
    boBotBars, boTopBars,
    'rgba(67,160,71,0.5)',
    'Plot Sentiment: Box Office Top 25% vs. Rest',
    'Top 25% BO (n=' + nBoTop + ')  vs.  bottom 75% (n=' + nBoBot + ')  |  threshold: $' + boThreshM + 'M',
    'leg-bobot', 'leg-botop'
  );
  setTimeout(() => {
    updateCardsComparison(
      'Bottom 75% box office', '#9a9a94', calcPcts(boBotBars),
      'Top 25% box office',   '#43A047', calcPcts(boTopBars)
    );
  }, 900);
}

function morphToImdb() {
  if (phase !== 'boxoffice' && phase !== 'oscar' && phase !== 'chart') return;
  phase = 'imdb';
  buildOverlayChart(
    imdbLowBars, imdbHighBars,
    'rgba(233,30,99,0.5)',
    'Plot Sentiment: High vs. Low IMDb Rating',
    'IMDb 7.5+ (n=' + nImdbHigh + ')  vs.  below 7.5 (n=' + nImdbLow + ')',
    'leg-imdblow', 'leg-imdbhigh'
  );
  setTimeout(() => {
    updateCardsComparison(
      'IMDb below 7.5', '#9a9a94', calcPcts(imdbLowBars),
      'IMDb 7.5+',      '#E91E63', calcPcts(imdbHighBars)
    );
  }, 900);
}

// ── Trigger animation (squares → chart) ──────────────────────────────────────
function triggerAnimation() {
  if (phase !== 'clusters') return;
  phase = 'animating';
  cancelAnimationFrame(rafId);
  const start = performance.now(), dur = 2600;
  function step(now) {
    const progress = (now - start) / dur;
    drawFrame(progress);
    if (progress < 1) {
      rafId = requestAnimationFrame(step);
    } else {
      phase = 'chart';
      document.getElementById('squares-stage').style.display = 'none';
      const cw = document.getElementById('chartWrap');
      cw.style.display = 'block';
      setTimeout(() => cw.classList.add('visible'), 30);
      buildChart();
    }
  }
  rafId = requestAnimationFrame(step);
}

// ── Init ──────────────────────────────────────────────────────────────────────
window.addEventListener('load', () => { resize(); idleLoop(); });
window.addEventListener('resize', resize);

window.addEventListener('message', e => {
  if (e.data === 'startAnimation') triggerAnimation();
  if (e.data === 'startOscar')     morphToOscar();
  if (e.data === 'startBoxOffice') morphToBoxOffice();
  if (e.data === 'startImdb') morphToImdb();
});

canvas.addEventListener('click', triggerAnimation);
document.getElementById('chartWrap').addEventListener('click', () => {
  if      (phase === 'chart')      morphToOscar();
  else if (phase === 'oscar')      morphToBoxOffice();
  else if (phase === 'boxoffice')  morphToImdb();
});
</script>
</body>
</html>
'''

html = (html_template
  .replace('__BARS_JSON__',      json.dumps(bar_data))
  .replace('__OSCAR_NO_BARS__',  json.dumps(oscar_no_bars))
  .replace('__OSCAR_YES_BARS__', json.dumps(oscar_yes_bars))
  .replace('__BO_BOT_BARS__',    json.dumps(bo_bot_bars))
  .replace('__BO_TOP_BARS__',    json.dumps(bo_top_bars))
  .replace('__NEG_PCT__',        str(neg_pct))
  .replace('__NEU_PCT__',        str(neu_pct))
  .replace('__POS_PCT__',        str(pos_pct))
  .replace('__TOTAL__',          str(len(df)))
  .replace('__N_OSCAR__',        str(n_oscar))
  .replace('__N_NON_OSCAR__',    str(n_non_oscar))
  .replace('__N_BO_TOP__',       str(n_bo_top))
  .replace('__N_BO_BOT__',       str(n_bo_bot))
  .replace('__BO_THRESH_M__',    str(bo_thresh_m))
  .replace('__IMDB_HIGH_BARS__', json.dumps(imdb_high_bars))
  .replace('__IMDB_LOW_BARS__',  json.dumps(imdb_low_bars))
  .replace('__N_IMDB_HIGH__',    str(n_imdb_high))
  .replace('__N_IMDB_LOW__',     str(n_imdb_low))
)

with open('sentiment_animation_initial.html', 'w', encoding='utf-8') as f:
    f.write(html)

### Final part of sentiment
- Note: Requires running the last python section (not the html)
- Note 2: I was not able to make it work, Claude decided to yap to much and consumed my tokens without an answer. As an alternative, I asked to describe what the code is supposed to do:
- *The section tries to add something else to give a final for the entire sentiment analysis part and connect with the interactive map, but it is not working*

**Python part:**

Uses already-computed bar data from previous analyses (overall, Oscar, box office, IMDb splits)
Injects all group data as JSON into the HTML template

**HTML file — what it shows:**

Title and subtitle visible immediately
A "sentiment fingerprint" chart: one horizontal stacked bar per group (All films, Non-Oscar, Oscar winners, Bottom 75% BO, Top 25% BO, IMDb below 7.5, IMDb 7.5+), each bar showing neg/neu/pos percentage split side by side for direct comparison
A tension paragraph: acknowledges that tone shifts across groups but never dramatically, and that sentiment alone doesn't explain success
Four question cards, each posing an open question that the interactive map could answer (Crime neighborhoods, Oscar locations, Box office patterns, IMDb geography)
A bridge block: icon + single line inviting the user to explore the map

**The intended progression:**

Each section appears one after another as the user interacts
The fingerprint chart is the key new visual — it's the first time all four analyses appear simultaneously, making comparison possible
The tension paragraph reframes the findings honestly: no strong conclusions
The question cards deliberately leave things open, transferring curiosity to the map
The bridge block closes the sentiment section and points forward

**The core purpose:**

Not to conclude, but to synthesize and provoke
The user leaves this section with questions rather than answers
Those questions are designed to be answerable on the interactive map that follows

In [10]:
html_conclusion = '''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Sentiment Conclusion</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    background: #0a0a0a;
    font-family: -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif;
    color: #f0ece4;
    padding: 1.5rem 2rem 2rem;
    min-height: 100vh;
  }
  .container { max-width: 680px; margin: 0 auto; }

  .section-label {
    font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase;
    color: #555550; margin-bottom: 0.5rem;
  }
  h2 { font-size: 20px; font-weight: 500; color: #f0ece4; margin-bottom: 0.35rem; }
  .subtitle { font-size: 14px; color: #aaa89f; margin-bottom: 1.5rem; line-height: 1.6; }

  /* ── Next button ── */
  #next-btn {
    display: inline-flex; align-items: center; gap: 8px;
    background: #1a1a1a; border: 1px solid #2a2a2a;
    color: #9a9a94; font-size: 13px;
    padding: 0.6rem 1.2rem; border-radius: 8px;
    cursor: pointer; margin-bottom: 2rem;
    transition: border-color 0.2s, color 0.2s;
    font-family: inherit;
  }
  #next-btn:hover { border-color: #444; color: #f0ece4; }
  #next-btn.hidden { display: none; }

  /* ── Progress dots ── */
  .progress {
    display: flex; gap: 6px; margin-bottom: 1.5rem;
  }
  .dot {
    width: 6px; height: 6px; border-radius: 50%;
    background: #222; transition: background 0.3s;
  }
  .dot.active { background: #555550; }
  .dot.done   { background: #2a2a2a; }

  /* ── Sections ── */
  .step-section {
    display: none;
    margin-bottom: 1.5rem;
    pointer-events: none;
  }
  .step-section.visible {
    opacity: 1; transform: translateY(0);
    pointer-events: auto;
  }

  /* ── Legend ── */
  .fp-legend { display: flex; gap: 16px; margin-bottom: 1.25rem; flex-wrap: wrap; }
  .fp-legend-item { display: flex; align-items: center; gap: 5px; font-size: 12px; color: #777770; }
  .fp-legend-dot { width: 10px; height: 10px; border-radius: 2px; }

  .fp-axis { display: grid; grid-template-columns: 160px 1fr 52px; gap: 12px; margin-bottom: 6px; }
  .fp-axis-labels { display: flex; justify-content: space-between; font-size: 10px; color: #333330; }

  .fp-row {
    display: grid; grid-template-columns: 160px 1fr 52px;
    align-items: center; gap: 12px; margin-bottom: 10px;
  }
  .fp-label { font-size: 12px; color: #9a9a94; text-align: right; line-height: 1.3; }
  .fp-label strong { display: block; font-size: 13px; color: #d0cdc6; font-weight: 500; }
  .fp-bar-track {
    height: 26px; border-radius: 3px; overflow: hidden;
    display: flex; background: #141414;
  }
  .fp-seg {
    height: 100%; display: flex; align-items: center; justify-content: center;
    font-size: 10px; font-weight: 500; color: rgba(255,255,255,0.8);
    min-width: 0; overflow: hidden; white-space: nowrap;
    width: 0%;
  }
  .fp-seg.neg { background: #c0543a; }
  .fp-seg.neu { background: #4a4a47; }
  .fp-seg.pos { background: #1a8a65; }
  .fp-n { font-size: 11px; color: #444440; white-space: nowrap; }
  .fp-divider-el { height: 1px; background: #1a1a1a; margin: 6px 0 12px; }

  /* ── Tension ── */
  .tension-block {
    background: #0f0f0f; border-left: 3px solid #2a2a2a;
    border-radius: 0 8px 8px 0; padding: 1rem 1.25rem;
  }
  .tension-block p { font-size: 13px; color: #777770; line-height: 1.7; }
  .tension-block strong { color: #aaa89f; }

  /* ── Cards ── */
  .cards-label {
    font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase;
    color: #555550; margin-bottom: 1rem;
  }
  .cards-grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 12px; }
  .q-card {
    background: #111; border: 1px solid #1e1e1e;
    border-radius: 12px; padding: 1.1rem 1.2rem;
  }
  .q-card:hover { border-color: #2a2a28; background: #141414; }
  .q-card-tag {
    font-size: 10px; font-weight: 600; letter-spacing: 0.08em;
    text-transform: uppercase; margin-bottom: 0.5rem;
    display: flex; align-items: center; gap: 6px;
  }
  .q-card-dot { width: 8px; height: 8px; border-radius: 50%; }
  .q-card-question { font-size: 13px; color: #aaa89f; line-height: 1.55; font-style: italic; }

  /* ── Bridge ── */
  .bridge-block { text-align: center; padding: 1.5rem 0 1rem; }
  .bridge-icon  { font-size: 28px; margin-bottom: 0.75rem; }
  .bridge-text  { font-size: 15px; color: #d0cdc6; font-weight: 500; margin-bottom: 0.4rem; }
  .bridge-sub   { font-size: 13px; color: #555550; line-height: 1.6; }
  .bridge-arrow {
    margin-top: 1rem; font-size: 18px; color: #333330;
    animation: bob 1.8s ease-in-out infinite;
  }
  @keyframes bob {
    0%,100% { transform: translateY(0); }
    50%      { transform: translateY(5px); }
  }
</style>
</head>
<body>
<div class="container">

  <p class="section-label">Sentiment Analysis — All Together</p>
  <h2>How does tone shift across each cut?</h2>
  <p class="subtitle">
    Oscar winners, box office hits, critically acclaimed films —
    each group has a different emotional profile. Here they are side by side.
  </p>

  <!-- Progress dots -->
  <div class="progress" id="progress"></div>

  <!-- Next button -->
  <button id="next-btn">Continue ↓</button>  

  <!-- Step 1: Fingerprint chart -->
  <div class="step-section" id="step-1">
    <div class="fp-legend">
      <span class="fp-legend-item"><span class="fp-legend-dot" style="background:#c0543a;"></span>Negative plots</span>
      <span class="fp-legend-item"><span class="fp-legend-dot" style="background:#4a4a47;"></span>Neutral</span>
      <span class="fp-legend-item"><span class="fp-legend-dot" style="background:#1a8a65;"></span>Positive plots</span>
    </div>
    <div class="fp-axis">
      <div></div>
      <div class="fp-axis-labels">
        <span>0%</span><span>25%</span><span>50%</span><span>75%</span><span>100%</span>
      </div>
      <div></div>
    </div>
    <div id="fp-rows"></div>
  </div>

  <!-- Step 2: Tension -->
  <div class="step-section" id="step-2">
    <div class="tension-block">
      <p>
        Tone shifts across groups, but <strong>never dramatically</strong>.
        Crime films score darkest, Musicals brightest —
        yet both have acclaimed, commercially successful titles in this dataset.
        <strong>Sentiment alone does not explain what makes a San Francisco film resonate.</strong>
        Something else is going on — and it might be written in the city itself.
      </p>
    </div>
  </div>

  <!-- Step 3: Question cards -->
  <div class="step-section" id="step-3">
    <p class="cards-label">Questions worth exploring</p>
    <div class="cards-grid" id="cards-grid"></div>
  </div>

  <!-- Step 4: Bridge -->
  <div class="step-section" id="step-4">
    <div class="bridge-block">
      <div class="bridge-icon">🗺️</div>
      <div class="bridge-text">One layer at a time was not enough.</div>
      <div class="bridge-sub">
        Explore filming locations, genres, ratings and sentiment<br>
        all together on the interactive map below.
      </div>
      <div class="bridge-arrow">↓</div>
    </div>
  </div>

</div>
<script>
const allBars    = __BARS_JSON__;
const oscarYes   = __OSCAR_YES_BARS__;
const oscarNo    = __OSCAR_NO_BARS__;
const boTop      = __BO_TOP_BARS__;
const boBot      = __BO_BOT_BARS__;
const imdbHigh   = __IMDB_HIGH_BARS__;
const imdbLow    = __IMDB_LOW_BARS__;
const nOscar     = __N_OSCAR__;
const nNonOscar  = __N_NON_OSCAR__;
const nBoTop     = __N_BO_TOP__;
const nBoBot     = __N_BO_BOT__;
const nImdbHigh  = __N_IMDB_HIGH__;
const nImdbLow   = __N_IMDB_LOW__;
const totalFilms = __TOTAL__;

let currentStep = 0;
const TOTAL_STEPS = 4;

function calcPcts(barsArr) {
  const t = barsArr.reduce((a,b) => a+b.y, 0);
  if (t === 0) return { neg:0, neu:0, pos:0 };
  let neg=0, neu=0, pos=0;
  barsArr.forEach(b => {
    if      (b.x < -0.05)  neg += b.y;
    else if (b.x <=  0.05) neu += b.y;
    else                   pos += b.y;
  });
  return {
    neg: Math.round(neg/t*100),
    neu: Math.round(neu/t*100),
    pos: Math.round(pos/t*100),
  };
}

const groups = [
  { label:'All SF films',       sub: totalFilms+' films', bars:allBars,  accent:'#f0ece4', divider:true  },
  { label:'Non-Oscar winners',  sub: nNonOscar+' films',  bars:oscarNo,  accent:'#9a9a94', divider:false },
  { label:'Oscar winners',      sub: nOscar+' films',     bars:oscarYes, accent:'#FFD700', divider:true  },
  { label:'Bottom 75% BO',      sub: nBoBot+' films',     bars:boBot,    accent:'#9a9a94', divider:false },
  { label:'Top 25% box office', sub: nBoTop+' films',     bars:boTop,    accent:'#43A047', divider:true  },
  { label:'IMDb below 7.5',     sub: nImdbLow+' films',   bars:imdbLow,  accent:'#9a9a94', divider:false },
  { label:'IMDb 7.5+',          sub: nImdbHigh+' films',  bars:imdbHigh, accent:'#E91E63', divider:false },
];

const qCards = [
  { tag:'Crime · Dark tone',    color:'#c0543a',
    question:'SF\'s darkest genre — does it concentrate in specific neighborhoods?' },
  { tag:'Oscar winners',        color:'#FFD700',
    question:'Only ' + nOscar + ' films won Oscars. Where were they filmed?' },
  { tag:'Top 25% box office',   color:'#43A047',
    question:'Top earners don\'t always have uplifting plots. What do they share?' },
  { tag:'IMDb 7.5+',            color:'#E91E63',
    question:'Acclaimed films spread across all tones. Does location play a role?' },
];

function buildRows() {
  const wrap = document.getElementById('fp-rows');
  if (!wrap) return;
  wrap.innerHTML = '';
  groups.forEach(g => {
    const p   = calcPcts(g.bars);
    const row = document.createElement('div');
    row.className = 'fp-row';
    row.innerHTML =
      '<div class="fp-label" style="color:' + g.accent + '">' +
        '<strong>' + g.label + '</strong>' + g.sub + '</div>' +
      '<div class="fp-bar-track">' +
        '<div class="fp-seg neg" data-target="' + p.neg + '" style="width:0%">' +
          (p.neg >= 12 ? p.neg+'%' : '') + '</div>' +
        '<div class="fp-seg neu" data-target="' + p.neu + '" style="width:0%">' +
          (p.neu >= 8  ? p.neu+'%' : '') + '</div>' +
        '<div class="fp-seg pos" data-target="' + p.pos + '" style="width:0%">' +
          (p.pos >= 12 ? p.pos+'%' : '') + '</div>' +
      '</div>' +
      '<div class="fp-n">n=' + g.bars.reduce((a,b)=>a+b.y,0) + '</div>';
    wrap.appendChild(row);
    if (g.divider) {
      const d = document.createElement('div');
      d.className = 'fp-divider-el';
      wrap.appendChild(d);
    }
  });
}

function animateBars() {
  document.querySelectorAll('.fp-row').forEach((row, ri) => {
    setTimeout(() => {
      const segs    = row.querySelectorAll('.fp-seg');
      const targets = Array.from(segs).map(s => parseFloat(s.dataset.target) || 0);
      const dur = 800, t0 = performance.now();
      function step(now) {
        const t = Math.min((now - t0) / dur, 1);
        const e = t < 0.5 ? 2*t*t : -1+(4-2*t)*t;
        segs.forEach((seg, i) => { seg.style.width = (targets[i] * e) + '%'; });
        if (t < 1) requestAnimationFrame(step);
      }
      requestAnimationFrame(step);
    }, ri * 100);
  });
}

function buildCards() {
  const grid = document.getElementById('cards-grid');
  if (!grid) return;
  grid.innerHTML = '';
  qCards.forEach(c => {
    const card = document.createElement('div');
    card.className = 'q-card';
    card.innerHTML =
      '<div class="q-card-tag" style="color:' + c.color + '">' +
        '<span class="q-card-dot" style="background:' + c.color + '"></span>' + c.tag + '</div>' +
      '<div class="q-card-question">' + c.question + '</div>';
    grid.appendChild(card);
  });
}

function updateProgress() {
  for (let i = 0; i < TOTAL_STEPS; i++) {
    const dot = document.getElementById('dot-' + i);
    if (!dot) continue;
    if (i < currentStep)        dot.className = 'dot done';
    else if (i === currentStep) dot.className = 'dot active';
    else                        dot.className = 'dot';
  }
}

function showStep(n) {
  const el = document.getElementById('step-' + n);
  if (!el) { console.log('step not found: ' + n); return; }
  // force display first, then opacity
  el.style.display   = 'block';
  el.style.opacity   = '0';
  el.style.transform = 'translateY(16px)';
  // force reflow
  void el.offsetHeight;
  // now animate
  el.style.transition  = 'opacity 0.5s ease, transform 0.5s ease';
  el.style.opacity     = '1';
  el.style.transform   = 'translateY(0)';
  el.style.pointerEvents = 'auto';

  if (n === 1) setTimeout(animateBars, 400);

  setTimeout(() => {
    el.scrollIntoView({ behavior: 'smooth', block: 'start' });
  }, 100);
}

function nextStep() {
  currentStep++;
  updateProgress();
  showStep(currentStep);
  if (currentStep >= TOTAL_STEPS) {
    const btn = document.getElementById('next-btn');
    if (btn) btn.style.display = 'none';
  }
}

// ── Init ──────────────────────────────────────────────────────────────────────
buildRows();
buildCards();

// Build progress dots
const progressWrap = document.getElementById('progress');
if (progressWrap) {
  for (let i = 0; i < TOTAL_STEPS; i++) {
    const dot = document.createElement('div');
    dot.className = 'dot';
    dot.id = 'dot-' + i;
    progressWrap.appendChild(dot);
  }
}

// Attach button
const btn = document.getElementById('next-btn');
if (btn) btn.addEventListener('click', nextStep);

// Show step 1 immediately
updateProgress();
setTimeout(nextStep, 300);
</script>
</body>
</html>
'''

html_conc = (html_conclusion
  .replace('__BARS_JSON__',      json.dumps(bar_data))
  .replace('__OSCAR_YES_BARS__', json.dumps(oscar_yes_bars))
  .replace('__OSCAR_NO_BARS__',  json.dumps(oscar_no_bars))
  .replace('__BO_TOP_BARS__',    json.dumps(bo_top_bars))
  .replace('__BO_BOT_BARS__',    json.dumps(bo_bot_bars))
  .replace('__IMDB_HIGH_BARS__', json.dumps(imdb_high_bars))
  .replace('__IMDB_LOW_BARS__',  json.dumps(imdb_low_bars))
  .replace('__N_OSCAR__',        str(n_oscar))
  .replace('__N_NON_OSCAR__',    str(n_non_oscar))
  .replace('__N_BO_TOP__',       str(n_bo_top))
  .replace('__N_BO_BOT__',       str(n_bo_bot))
  .replace('__N_IMDB_HIGH__',    str(n_imdb_high))
  .replace('__N_IMDB_LOW__',     str(n_imdb_low))
  .replace('__TOTAL__',          str(len(df)))
)

with open('sentiment_conclusion.html', 'w', encoding='utf-8') as f:
    f.write(html_conc)

print('Saved -> sentiment_conclusion.html')

Saved -> sentiment_conclusion.html


- Info about the neutral movies (Additional)

In [27]:
# Print the 5 most neutral movies (compound between -0.05 and 0.05)
neutral_movies = df[df['compound'].between(-0.05, 0.05)].sort_values('compound')
print('\n5 most neutral films:')
print(neutral_movies[['Title', 'compound', 'Genres']].head(5).to_string(index=False))



5 most neutral films:
                 Title  compound                    Genres
        Final Analysis   -0.0387           Drama, Thriller
         True Believer   -0.0227     Crime, Drama, Mystery
             The Doors   -0.0226   Biography, Drama, Music
                Babies    0.0000               Documentary
Fat Man and Little Boy    0.0000 Biography, Drama, History
